# Module 04 — Control Flow, Functions, and Scope

## Exercise 04.3 — Four things built with closures only

No classes. The constraint is the exercise: everything here can be done with a
class, and doing it with closures first is what makes decorators (Module 15)
feel like a natural next step rather than magic syntax.
Run:  python ex03_closures.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. Loops, and the `else` nobody expects

In [ ]:
for item in collection:      # iterates ANYTHING iterable (Module 14)
    ...

for i, item in enumerate(collection, start=1):
    ...

for a, b in zip(xs, ys, strict=True):    # strict=True is 3.10+ and you want it
    ...

for key, value in mapping.items():
    ...

`zip(strict=True)` raises if the iterables have different lengths. Without it,
`zip` silently stops at the shortest, which has hidden many data bugs. Default
to `strict=True` unless truncation is genuinely intended.

### `for ... else`

The `else` clause runs **if the loop completed without `break`**. It is not "if
the loop body never ran".

In [ ]:
for user in users:
    if user.is_admin:
        print("found an admin")
        break
else:
    print("no admin found")        # runs only if we never broke out

Read `else` here as `nobreak` and it becomes obvious. It exists to remove the
`found = False` flag variable:

In [ ]:
found = False                       # the pattern for ... else replaces
for user in users:
    if user.is_admin:
        found = True
        break
if not found:
    ...

It is rare in real code, and it is on every Python quiz.

### Loop control

```text
break        # exit the innermost loop
continue     # next iteration
```


There is no labelled break. To exit nested loops, either extract the loops into
a function and `return`, or use a flag, or iterate a product:

In [ ]:
from itertools import product
for i, j in product(range(n), range(m)):
    if done(i, j):
        break                       # one loop, so one break is enough

Extracting to a function is almost always the cleanest of the three.

### Do not mutate what you are iterating

In [ ]:
for x in items:
    if pred(x):
        items.remove(x)             # silently skips elements (Module 02, q11)

items = [x for x in items if not pred(x)]      # correct
items[:] = [x for x in items if not pred(x)]   # correct, and in place

---

## Concept 3. `match`: structural pattern matching, not a switch

`match` (3.10+) destructures values. Using it as a C-style switch wastes it.

In [ ]:
match command.split():
    case ["go", direction]:
        move(direction)
    case ["take", *items]:                 # capture the rest
        for item in items:
            take(item)
    case ["quit" | "exit"]:                # alternatives
        raise SystemExit
    case []:
        print("say something")
    case _:                                 # the default; _ matches anything
        print(f"unknown: {command}")

It matches structure, types, and attributes:

In [ ]:
match event:
    case {"type": "click", "pos": (x, y)}:          # dict + tuple shape
        handle_click(x, y)
    case {"type": "key", "code": int() as code}:    # type check + capture
        handle_key(code)
    case Point(x=0, y=0):                            # class patterns
        print("origin")
    case Point(x=x, y=y) if x == y:                  # a guard
        print("diagonal")

Two traps:

**A bare name is a capture, not a comparison.**

```text
case OK:              # binds anything to the name OK. Always matches!
case Status.OK:       # a dotted name IS compared. This is what you meant.
```


This is the number one `match` bug. Any pattern that is a plain identifier
captures; only dotted names, literals, and class patterns compare.

**Class patterns need `__match_args__`** for positional matching, which
`@dataclass` provides automatically (Module 11).

When is `match` worth it? When you are destructuring nested data — parsing,
protocol handling, AST walking, event dispatch. For dispatching on a single
value, a dict of functions is clearer and faster.

---

## Concept 4. Functions: the six kinds of parameter

In [ ]:
def f(pos_only, /, standard, *args, kw_only, **kwargs):
    ...

| Kind | Declared | Called as |
|---|---|---|
| Positional-only | before `/` | `f(1)` only |
| Positional-or-keyword | between `/` and `*` | `f(1)` or `f(standard=1)` |
| Var-positional | `*args` | extra positionals collected into a tuple |
| Keyword-only | after `*` | `f(kw_only=1)` only |
| Var-keyword | `**kwargs` | extra keywords collected into a dict |

In [ ]:
def connect(host, port=5432, /, *, timeout=30, retries=3, **options):
    ...

connect("db", 5432, timeout=10, ssl=True)      # ok
connect(host="db")                              # TypeError: host is positional-only
connect("db", 5432, 10)                         # TypeError: timeout is keyword-only

**Why bother?**

- `/` (positional-only) frees you to rename parameters later without breaking
  callers. The standard library uses it heavily for exactly this reason.
- `*` (keyword-only) forces call sites to be readable. `resize(img, 800, 600,
  True, False)` is unreadable; `resize(img, width=800, height=600,
  preserve_aspect=True, upscale=False)` is not.

**Rule of thumb: any boolean parameter should be keyword-only.** A bare `True`
at a call site carries no information.

### Arguments are unpacked, not copied

In [ ]:
args = (1, 2)
kwargs = {"c": 3}
f(*args, **kwargs)          # equivalent to f(1, 2, c=3)

### The mutable default, again

```text
def f(items=[]):            # WRONG -- evaluated once at def time (Module 02)
def f(items=None):          # right
    if items is None:
        items = []
```


Same for `{}`, `set()`, `datetime.now()`, and any expression whose value should
be per-call. `ruff` rule `B006` catches it.

### Functions are objects

In [ ]:
def greet(name): return f"hi {name}"

greet.__name__          # 'greet'
greet.__doc__           # the docstring
greet.__defaults__      # the default values tuple
greet.__annotations__   # the type hints, as a dict

handlers = {"greet": greet}          # store them
def apply(fn, x): return fn(x)       # pass them
def make(): return greet             # return them

This is what makes decorators, callbacks, and higher-order functions possible,
and it is the subject of Module 15.

---

## Concept 5. Scope: LEGB

Name lookup walks four scopes, in order:

```text
L  Local        the current function's own names
E  Enclosing    any enclosing function's names (closures)
G  Global       the module's top-level names
B  Builtins     print, len, list, ...
```


In [ ]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print(x)        # local
    inner()
    print(x)            # enclosing
outer()
print(x)                # global

### Assignment makes a name local for the whole function

This is the rule that produces the most confusing error in the language:

In [ ]:
counter = 0

def increment():
    counter += 1        # UnboundLocalError: local variable 'counter'
                        # referenced before assignment

The compiler scans the function body *before* it runs. It sees `counter` being
assigned somewhere in the body, so `counter` is a **local** for the entire
function — including on the line that reads it, which happens before any write.
Reading it there is reading an unassigned local.

Note the asymmetry that makes this so confusing:

In [ ]:
def read_only():
    print(counter)      # fine -- no assignment in this body, so it is global

def mutate_ok():
    items.append(1)     # fine -- MUTATION is not assignment

The fixes, in order of preference:

In [ ]:
def increment(counter: int) -> int:      # 1. best: take it in, hand it back
    return counter + 1

class Counter:                            # 2. state belongs in an object
    def __init__(self): self.n = 0
    def increment(self): self.n += 1

def increment():                          # 3. last resort
    global counter
    counter += 1

`global` is almost always a design smell. It makes a function's behaviour depend
on invisible state and makes it untestable in isolation.

### `nonlocal` for closures

In [ ]:
def make_counter():
    count = 0
    def increment():
        nonlocal count       # rebind the ENCLOSING count, not a new local
        count += 1
        return count
    return increment

c = make_counter()
c(); c(); c()          # 1, 2, 3

`global` reaches the module scope. `nonlocal` reaches the nearest enclosing
*function* scope. Neither reaches a class body.

### Comprehensions have their own scope

In [ ]:
i = "untouched"
squares = [i * i for i in range(5)]
print(i)                # 'untouched' -- the loop variable did not leak

True since Python 3. A plain `for` loop *does* leak its variable; a comprehension
does not.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Statements and expressions
- Section 2: Loops, and the `else` nobody expects
- Section 3: `match`: structural pattern matching, not a switch
- Section 4: Functions: the six kinds of parameter
- Section 5: Scope: LEGB
- Section 6: Closures
- Section 7: Type hints

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import time
from collections.abc import Callable
from typing import Any


# TODO 1 -----------------------------------------------------------------------

---

## `make_counter`

Return (increment, reset).

In [ ]:
def make_counter(start: int = 0, step: int = 1) -> tuple[Callable[[], int],
                                                          Callable[[], None]]:
    """Return (increment, reset).

    increment() adds step to the internal count and returns the NEW value.
    reset() sets it back to start.

    Both closures must share ONE count. That sharing is the whole point --
    write it, then inspect increment.__closure__ to see the cell they share.
    """
    raise NotImplementedError

---

## `memoize`

Return a wrapper that caches results by arguments.

In [ ]:
def memoize(fn: Callable[..., Any]) -> Callable[..., Any]:
    """Return a wrapper that caches results by arguments.

    Requirements:
      - cache hits must not call fn again (prove it with a call counter)
      - keyword arguments must be part of the key
      - the wrapper must expose .cache_info() returning (hits, misses, size)
      - the wrapper must expose .cache_clear()

    Then answer, in a comment:
      - what happens if an argument is unhashable? What SHOULD happen?
      - f(1) and f(1.0) and f(True): same cache entry or different? Why?
      - why is caching a function with side effects a bug?

    functools.lru_cache does all this properly. Writing it once tells you what
    it is doing and when it is unsafe.
    """
    raise NotImplementedError

---

## `make_event_bus`

Return (subscribe, emit).

In [ ]:
def make_event_bus() -> tuple[Callable[[str, Callable[..., Any]], Callable[[], None]],
                              Callable[..., list[Any]]]:
    """Return (subscribe, emit).

    subscribe(event_name, handler) registers a handler and returns an
    UNSUBSCRIBE function -- a closure over the registration, so the caller does
    not need to hold an ID or pass the handler back.

    emit(event_name, *args) calls every handler for that event, in registration
    order, and returns their results.

    Requirements:
      - a handler that raises must not prevent later handlers from running
      - unsubscribing twice must be safe
      - unsubscribing during an emit must not corrupt the iteration
        (that last one is a real bug in many hand-rolled event systems --
         think about Module 02's "never mutate while iterating")
    """
    raise NotImplementedError

---

## `with_retry`

A decorator FACTORY: with_retry(attempts=5) returns a decorator.

In [ ]:
def with_retry(
    attempts: int = 3,
    delay: float = 0.01,
    backoff: float = 2.0,
    exceptions: tuple[type[BaseException], ...] = (Exception,),
) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """A decorator FACTORY: with_retry(attempts=5) returns a decorator.

    Three nested functions. Getting this right is the whole skill that Module
    15 formalises:

        with_retry(...)  -> decorator  -> wrapper  -> the actual call

    Requirements:
      - retry only on the listed exception types; everything else propagates
        immediately
      - sleep `delay` after a failure, multiplying by `backoff` each time
      - after the last attempt, re-raise the LAST exception (not a new one)
      - the wrapper must keep the original function's __name__ and __doc__
        (do it by hand first, then look at functools.wraps and see that it is
         exactly what you just wrote)

    Then answer: why is retrying a non-idempotent operation dangerous, and what
    would you add to make it safe? (Module 33 answers this properly.)
    """
    raise NotImplementedError

---

## `test_counter`

_test counter_

In [ ]:
def test_counter() -> None:
    inc, reset = make_counter(start=10, step=5)
    assert inc() == 15
    assert inc() == 20
    reset()
    assert inc() == 15

    a_inc, _ = make_counter()
    b_inc, _ = make_counter()
    a_inc()
    assert b_inc() == 1, "separate counters must not share a cell"

---

## `test_memoize`

_test memoize_

In [ ]:
def test_memoize() -> None:
    calls = 0

    @memoize
    def slow(n: int, offset: int = 0) -> int:
        nonlocal calls
        calls += 1
        return n * 2 + offset

    assert slow(5) == 10
    assert slow(5) == 10
    assert calls == 1, "second call should have hit the cache"
    assert slow(5, offset=1) == 11
    assert calls == 2, "keyword arguments must be part of the key"

    hits, misses, size = slow.cache_info()
    assert (hits, misses, size) == (1, 2, 2), (hits, misses, size)
    slow.cache_clear()
    assert slow.cache_info()[2] == 0

---

## `test_event_bus`

_test event bus_

In [ ]:
def test_event_bus() -> None:
    subscribe, emit = make_event_bus()
    seen: list[str] = []

    un_a = subscribe("tick", lambda: seen.append("a"))
    subscribe("tick", lambda: seen.append("b"))

    def explodes() -> None:
        raise RuntimeError("handler failure")

    subscribe("tick", explodes)
    subscribe("tick", lambda: seen.append("c"))

    emit("tick")
    assert seen == ["a", "b", "c"], f"a raising handler broke the bus: {seen}"

    seen.clear()
    un_a()
    un_a()  # must be safe
    emit("tick")
    assert "a" not in seen

    emit("no-such-event")  # must not raise

---

## `test_retry`

_test retry_

In [ ]:
def test_retry() -> None:
    attempts = 0

    @with_retry(attempts=3, delay=0.001)
    def flaky() -> str:
        """docstring preserved?"""
        nonlocal attempts
        attempts += 1
        if attempts < 3:
            raise ConnectionError("boom")
        return "ok"

    assert flaky() == "ok"
    assert attempts == 3
    assert flaky.__name__ == "flaky", "wrapper lost the function's name"
    assert flaky.__doc__ == "docstring preserved?"

    @with_retry(attempts=2, delay=0.001, exceptions=(ConnectionError,))
    def wrong_error() -> None:
        raise ValueError("not retryable")

    start = time.perf_counter()
    try:
        wrong_error()
    except ValueError:
        pass
    else:
        raise AssertionError("unlisted exceptions must propagate")
    assert time.perf_counter() - start < 0.05, "it retried something it should not"

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    for t in tests:
        t()
        print(f"  PASS  {t.__name__}")
    print(f"\n{len(tests)} tests passed")

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.